In [2]:
import torch
import torch.nn as nn

In [15]:
class BiRNN(nn.Module):  
    def __init__(self, input_size, hidden_size, num_layers):  
        super(BiRNN, self).__init__()  
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, 
                          batch_first=True, bidirectional=True)  
        self.fc = nn.Linear(hidden_size * 2, 1)  # 考虑到双向RNN的隐藏状态是两倍  
  
    def forward(self, x, hx):  
        out, hx = self.rnn(x, hx)  
        h_out = out[:, -1, :]  
        output = self.fc(h_out)  
        return output, hx

In [16]:
import torch  
import torch.nn as nn  
  
# 定义模型参数  
input_size = 10  # 输入数据的大小  
hidden_size = 20  # 隐藏层的大小  
num_layers = 2  # RNN的层数  
batch_size = 32  # 批次大小  
seq_length = 10  # 序列长度  
  
# 创建BiRNN模型实例  
model = BiRNN(input_size, hidden_size, num_layers)  
print(model)  
  
# 生成随机输入数据和初始隐藏状态（这里仅为示例，实际使用时需要替换为实际数据）  
input_data = torch.randn(batch_size, seq_length, input_size)  
initial_hidden = (torch.randn(2, num_layers * 1, hidden_size),  # 正向隐藏状态  
                 torch.randn(2, num_layers * 1, hidden_size))  # 反向隐藏状态  
# 当rnn为RNN时
initial_hidden = torch.randn(num_layers * 2,batch_size, hidden_size)

# 当rnn为LSTM时
# initial_hidden = (torch.zeros(num_layers * 2, batch_size, hidden_size),  # 正向隐藏状态  
#                   torch.zeros(num_layers * 2, batch_size, hidden_size)
#                  ) # 当rnn为LSTM时

# 前向传播计算输出结果和隐藏状态  
output, hx = model(input_data, initial_hidden)  
print("Output shape:", output.shape)  # 输出结果的形状为(batch_size, hidden_size * 2)  
print("Next hidden shape:", hx[0].shape, hx[1].shape)  # 下一个隐藏状态的形状为(2, num_layers * num_directions, hidden_size)

BiRNN(
  (rnn): RNN(10, 20, num_layers=2, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=40, out_features=1, bias=True)
)
Output shape: torch.Size([32, 1])
Next hidden shape: torch.Size([32, 20]) torch.Size([32, 20])


### LSTM 与 RNN 可能稍有不同

In [32]:
## -------------------------------- RNN ------------------------------------- ##
# 定义模型参数  
input_size = 10    # 输入数据的大小  
hidden_size = 20   # 隐藏层的大小  
num_layers = 1     # RNN的层数  
batch_size = 32    # 批次大小  
seq_length = 10    # 序列长度  
is_bidirect = False # True:双向  False:单向


rnn = nn.RNN(input_size, hidden_size, 
                  num_layers, batch_first=True, bidirectional=is_bidirect) 
input_data = torch.randn(batch_size, seq_length, input_size) 
#                             因为双层的原因 所以*2
factor = 2 if is_bidirect else 1
initial_hidden = torch.randn(num_layers * factor, batch_size, hidden_size)
output, hx = rnn(input_data, initial_hidden)
# 因为双向的原因 才 *2
print(output.shape) # torch.Size([batch=32, input_size=10, hidden_size*2=40]) 
print(hx.shape) # torch.Size([num_layers*2=4, batch_size=32, hidden_size=20])

torch.Size([32, 10, 20])
torch.Size([1, 32, 20])


In [34]:
## -------------------------------- LSTM ------------------------------------- ##
# 定义模型参数  
input_size = 10    # 输入数据的大小  
hidden_size = 20   # 隐藏层的大小  
num_layers = 2     # RNN的层数  
batch_size = 32    # 批次大小  
seq_length = 10    # 序列长度  
is_bidirect = True # True:双向  False:单向

lstm = nn.LSTM(input_size, hidden_size, 
                  num_layers, batch_first=True, bidirectional=is_bidirect) 
input_data = torch.randn(batch_size, seq_length, input_size) 
#                             因为双层的原因 所以*2
factor = 2 if is_bidirect else 1
initial_h = torch.randn(num_layers * factor, batch_size, hidden_size)
initial_c = torch.randn(num_layers * factor, batch_size, hidden_size)

output, hx = lstm(input_data, (initial_h,initial_c))
# 因为双向的原因 才 *2
print(output.shape) # torch.Size([batch=32, input_size=10, hidden_size*2=40]) 
print(hx[0].shape) # torch.Size([num_layers*2=4, batch_size=32, hidden_size=20])
print(hx[1].shape) # torch.Size([num_layers*2=4, batch_size=32, hidden_size=20])

torch.Size([32, 10, 40])
torch.Size([4, 32, 20])
torch.Size([4, 32, 20])
